In [1]:
!pip install numpy scipy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 62.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 132.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 122.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 108.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [matplotlib]9 [matplotlib]


In [2]:
import numpy as np
import scipy 
import matplotlib.pyplot as plt

$$
\begin{aligned}
\frac{dS}{dt} &= -\beta S I, \\
\frac{dI}{dt} &= \beta S I - \gamma I, \\
\frac{dR}{dt} &= \gamma I,
\end{aligned}
$$

where:

- $S$ is the number of *susceptible* people (those at risk of infection),
- $I$ is the number of *infected* people (those currently infectious),
- $R$ is the number of *recovered* people (those who have recovered and are immune),
- $\beta$ is the *transmission rate* (per person per day), and
- $\gamma$ is the *recovery rate* (per day).

In [25]:
# modeling
def sir_model(t, y, beta=0.125*1e-3, gamma=0.1):
    S, I, R = y
    dSdt = -beta * S * I
    dIdt = beta * S * I - gamma * I
    dRdt = gamma * I
    return np.array([dSdt, dIdt, dRdt])

# numerical solvers for the SIR model
def euler(f, y, t, beta=0.125*1e-3, gamma=0.1):
    for i in range(1, len(t)):
        y[i] = y[i-1] + dt * f(t[i-1], y[i-1], beta=beta, gamma=gamma)
    return t, y


def runge_kutta(f, y, t, beta=0.125*1e-3, gamma=0.1):
    for i in range(1, len(t)):
        dt = t[i] - t[i-1]
        k1 = f(t[i-1], y[i-1], beta=beta, gamma=gamma)
        k2 = f(t[i-1] + dt/2, y[i-1] + dt/2 * k1, beta=beta, gamma=gamma)
        k3 = f(t[i-1] + dt/2, y[i-1] + dt/2 * k2, beta=beta, gamma=gamma)
        k4 = f(t[i-1] + dt, y[i-1] + dt * k3, beta=beta, gamma=gamma)
        y[i] = y[i-1] + dt/6 * (k1 + 2*k2 + 2*k3 + k4)
    return t, y

# custom solver
def sir_solver(f, y0, days, dt, method="euler", beta=0.125*1e-3, gamma=0.1):
    t = np.arange(0, days + dt, dt)
    y = np.zeros((len(t), len(y0)))
    # initial population values
    y[0] = y0
    
    if method == "euler":
        return euler(f, y, t, beta=beta, gamma=gamma)
    elif method == "runge_kutta":
        return runge_kutta(f, y, t, beta=beta, gamma=gamma)

In [28]:
# validation
from scipy.integrate import solve_ivp

# Constants
y0 = [1995, 5, 0]
days = 160
dt = 0.1

# Euler
y_euler = sir_solver(sir_model, y0, days, dt, method="euler")
# Runge-Kutta
y_rk = sir_solver(sir_model, y0, days, dt, method="runge_kutta")
y_scipy = solve_ivp(sir_model, [0, days], y0, t_eval=np.arange(0, days + dt, dt), rtol=1e-8, atol=1e-8)
y_ref = y_scipy.y.T

print("==== Euler vs Reference ====")
diff_euler = np.abs(y_euler[1] - y_ref)
print(diff_euler)
print(np.max(diff_euler))

print("\n==== Runge-Kutta vs Reference ====")
diff_rk = np.abs(y_rk[1] - y_ref)
print(diff_rk)
print(np.max(diff_rk))

==== Euler vs Reference ====
[[0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [9.31926044e-04 5.56635354e-04 3.75290690e-04]
 [1.89147886e-03 1.12972301e-03 7.61755850e-04]
 ...
 [1.26677678e+00 6.35179952e-03 1.27312858e+00]
 [1.26675861e+00 6.31417266e-03 1.27307278e+00]
 [1.26674054e+00 6.27675627e-03 1.27301730e+00]]
10.716398681755663

==== Runge-Kutta vs Reference ====
[[0.00000000e+00 0.00000000e+00 0.00000000e+00]
 [5.43423084e-11 3.26014771e-11 2.14396764e-11]
 [2.21257324e-09 1.28956401e-09 9.22634288e-10]
 ...
 [3.04528271e-07 6.27795624e-09 2.98251280e-07]
 [3.04544216e-07 6.23130902e-09 2.98314035e-07]
 [3.04559990e-07 6.18511606e-09 2.98375880e-07]]
1.9141509937981027e-05


Based on the comparison above, Runge-Kutta will be used for higher accuracy. 